<a href="https://colab.research.google.com/github/RitvikM29/Deep_Learning_Codes/blob/main/Cancer_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Load dataset and preprocessing data**



In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# Placeholder for your actual file loading
df = pd.read_csv('breast-cancer.csv')
# Drop ID column
df = df.drop(columns=["id"])

# Encode diagnosis: M -> 1, B -> 0
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

# Split features and label
X = df.drop(columns=["diagnosis"]).values   # 30 features
y = df["diagnosis"].values.reshape(-1, 1)

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


**Single Perceptron — Sigmoid Activation**

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
def train_sigmoid_perceptron(X, y, lr=0.01, epochs=1000):
    w = np.zeros((X.shape[1], 1))
    b = 0

    for _ in range(epochs):
        z = np.dot(X, w) + b
        y_hat = sigmoid(z)

        dw = np.dot(X.T, (y_hat - y)) / len(y)
        db = np.mean(y_hat - y)

        w -= lr * dw
        b -= lr * db

    return w, b


In [ ]:
def evaluate_sigmoid(X, y, w, b):
    y_pred = sigmoid(np.dot(X, w) + b)
    y_class = (y_pred >= 0.5).astype(int)

    acc = np.mean(y_class == y)
    error = np.mean((y_pred - y) ** 2)

    return acc, error


In [ ]:
w1, b1 = train_sigmoid_perceptron(X_train, y_train)
train_acc1, train_err1 = evaluate_sigmoid(X_train, y_train, w1, b1)
test_acc1, test_err1 = evaluate_sigmoid(X_test, y_test, w1, b1)


**Single Perceptron — Threshold Function**

In [ ]:
def step(z):
    return (z >= 0).astype(int)


In [ ]:
def train_threshold_perceptron(X, y, lr=0.01, epochs=1000):
    w = np.zeros((X.shape[1], 1))
    b = 0

    for _ in range(epochs):
        z = np.dot(X, w) + b
        y_hat = step(z)

        w += lr * np.dot(X.T, (y - y_hat))
        b += lr * np.sum(y - y_hat)

    return w, b


In [ ]:
def evaluate_threshold(X, y, w, b):
    y_pred = step(np.dot(X, w) + b)
    acc = np.mean(y_pred == y)
    error = 1 - acc
    return acc, error

In [ ]:
w2, b2 = train_threshold_perceptron(X_train, y_train)
train_acc2, train_err2 = evaluate_threshold(X_train, y_train, w2, b2)
test_acc2, test_err2 = evaluate_threshold(X_test, y_test, w2, b2)

**One Hidden Layer NN**

In [ ]:
def train_mlp(X, y, lr=0.1, epochs=5000):
    n_features = X.shape[1]

    W1 = np.random.randn(n_features, 12) * 0.01
    b1 = np.zeros((1, 12))
    W2 = np.random.randn(12, 1) * 0.01
    b2 = np.zeros((1, 1))

    for _ in range(epochs):
        Z1 = np.dot(X, W1) + b1
        A1 = sigmoid(Z1)

        Z2 = np.dot(A1, W2) + b2
        A2 = sigmoid(Z2)

        dZ2 = A2 - y
        dW2 = np.dot(A1.T, dZ2) / len(y)
        db2 = np.mean(dZ2, axis=0)

        dA1 = np.dot(dZ2, W2.T)
        dZ1 = dA1 * A1 * (1 - A1)
        dW1 = np.dot(X.T, dZ1) / len(y)
        db1 = np.mean(dZ1, axis=0)

        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2

    return W1, b1, W2, b2


In [ ]:
def evaluate_mlp(X, y, W1, b1, W2, b2):
    A1 = sigmoid(np.dot(X, W1) + b1)
    A2 = sigmoid(np.dot(A1, W2) + b2)

    y_class = (A2 >= 0.5).astype(int)
    acc = np.mean(y_class == y)
    error = np.mean((A2 - y) ** 2)

    return acc, error


In [ ]:
W1, b1, W2, b2 = train_mlp(X_train, y_train)
train_acc3, train_err3 = evaluate_mlp(X_train, y_train, W1, b1, W2, b2)
test_acc3, test_err3 = evaluate_mlp(X_test, y_test, W1, b1, W2, b2)


In [ ]:
print("\t\t\tTrain Accuracy\tTest Accuracy\tTrain Error\tTest Error")
print("Sigmoid Perceptron:", train_acc1, test_acc1, train_err1, test_err1)
print("Threshold Perceptron:", train_acc2, test_acc2, train_err2, test_err2)
print("1-Hidden Layer NN:", train_acc3, test_acc3, train_err3, test_err3)


			Train Accuracy	Test Accuracy	Train Error	Test Error
Sigmoid Perceptron: 0.9824175824175824 0.9824561403508771 0.025960036250546895 0.020654928790741607
Threshold Perceptron: 1.0 0.9385964912280702 0.0 0.06140350877192979
1-Hidden Layer NN: 0.989010989010989 0.9824561403508771 0.009671286074823437 0.015034826587060613
